# Solar asset assessment — India

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/solar-asset-assessment/blob/main/notebooks/solar_asset_assessment.ipynb)

Takes every utility-scale solar plant in India from the **WRI Global Power
Plant Database** and tests the reported record against what satellites
observed at the site: when it was actually built, how the solar resource
compares with the 1991–2020 baseline, how hot it runs, whether vegetation is
creeping in, and whether it sits in a drainage line.

Five checks, published thresholds, one card per asset, plus a portfolio page.

**Runtime:** about 20–40 minutes for 25 assets. Everything is free data.

**You need:** a Google account with Earth Engine access, registered to a
Google Cloud project. Earth Engine's free tier is non-commercial.

---


## 1. Setup

Clones the repo so the notebook uses the same code as the tests, then
authenticates Earth Engine. The `ee.Authenticate()` step opens a consent
link the first time; paste the token back into the prompt.


In [ ]:
REPO_URL = 'https://github.com/YOUR-GITHUB-USERNAME/solar-asset-assessment.git'
EE_PROJECT = ''  # <-- your Google Cloud project id, e.g. 'ee-yourname'
N_ASSETS   = 25  # raise once you have seen it work end to end

import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('solar-asset-assessment'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL], check=True)

if os.path.exists('solar-asset-assessment'):
    os.chdir('solar-asset-assessment')
sys.path.insert(0, os.getcwd())
print('working in', os.getcwd())


In [ ]:
!pip -q install earthengine-api pandas


In [ ]:
import ee
from src import assess, card, site, config as C

assert EE_PROJECT, 'Set EE_PROJECT to your Google Cloud project id first.'

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print('Earth Engine ready:', ee.Number(1).add(1).getInfo() == 2)


## 2. Verify the rubric before running it

The scoring logic is pure Python with no Earth Engine in it, so it can be
tested in seconds. Run this before spending compute: if the rubric is wrong,
the satellite data will not save it.


In [ ]:
!pip -q install pytest && python -m pytest -q


## 3. Pick the assets

WRI is open, widely cited, and partly stale — which is the point. A reported
record is exactly what we want to test.


In [ ]:
assets = assess.load_assets(limit=N_ASSETS)
print(f'{len(assets)} assets, {assets.capacity_mw.sum():,.0f} MW total')
assets.head(10)


## 4. Run the assessment

One pass per asset: Dynamic World build history, ERA5-Land resource and heat,
Sentinel-2 NDVI trend, JRC surface water, Copernicus DEM terrain. Failures are
caught per asset and recorded on the card rather than ending the run.


In [ ]:
records = assess.assess_portfolio(assets)
paths = assess.save_results(records)
print(paths)


## 5. Look at the results


In [ ]:
import pandas as pd
df = pd.read_csv('outputs/results.csv')
cols = ['name', 'capacity_mw', 'commissioning_year', 'observed_build_month',
        'build_delay_m', 'score', 'traffic_light', 'evidence_completeness']
df[cols].sort_values('score', ascending=False)


In [ ]:
print(df.traffic_light.value_counts().to_string())
print()
print('mean evidence resolved: %.0f%%' % (df.evidence_completeness.mean()*100))
print()
for c in [c for c in df.columns if c.startswith('flag_')]:
    print(f'{c[5:]:<26} {int(df[c].sum())} of {len(df)}')


## 6. Build the published view

Writes one card per asset into `docs/cards/` and the portfolio page to
`docs/index.html`. Commit those and GitHub Pages serves them.


In [ ]:
import datetime as dt
as_of = dt.date.today().isoformat()
card_paths = card.write_cards(records)
index_path = site.write_site(records, as_of)
print(f'{len(card_paths)} cards + {index_path}')


In [ ]:
from IPython.display import IFrame, display
display(IFrame('docs/index.html', width='100%', height=700))


## 7. Commit the outputs

Set a token with `repo` scope, or skip this and download `docs/` and
`outputs/` from the Colab file browser and commit them locally.


In [ ]:
# from google.colab import userdata
# TOKEN = userdata.get('GITHUB_TOKEN')
# !git config user.email 'you@example.com'
# !git config user.name  'Your Name'
# !git add docs outputs
# !git commit -m 'Add assessment run outputs'
# !git push https://$TOKEN@github.com/YOUR-GITHUB-USERNAME/solar-asset-assessment.git main


---

## What this does not do

- The footprint is a capacity-scaled circle, not a segmented panel array. It
  over-includes surrounding land, which dilutes every area statistic.
- Height above nearest drainage is approximated from a DEM neighbourhood
  minimum. It is a screening proxy, not a hydraulic flood model.
- A flag is a reason to look, never a finding. Nothing here is evidence of
  misreporting by any operator.
- Unresolved checks score as clear, so a score is a lower bound. Every card
  states how much of the rubric was resolved.
- Earth Engine's free tier is non-commercial. This is a personal methodology
  prototype.
